# Preparação da base analítica

Como reunir pedidos, clientes, itens e avaliações sem duplicar pedidos? Prepara e valida as bases detalhadas para as análises.

In [ ]:
# Importa o pandas e permite visualizar todas as colunas dos DataFrames.
import pandas as pd
pd.set_option('display.max_columns', None)


In [ ]:
from pathlib import Path

# Procura a raiz a partir da pasta atual; funciona também nas cópias em outputs/notebooks.
RAIZ_PROJETO = next(
    (pasta for pasta in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (pasta / 'executar_pipeline.py').is_file() and (pasta / 'src').is_dir()),
    None
)
if RAIZ_PROJETO is None:
    raise FileNotFoundError('Execute o notebook dentro da pasta do projeto ou de uma subpasta.')

# Carrega as tabelas originais sem modificar os arquivos da pasta raw.
CAMINHO_DADOS = RAIZ_PROJETO / 'data' / 'raw'

df_order_reviews_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_order_reviews_dataset.csv')
df_customers_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_customers_dataset.csv')
df_order_items_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_order_items_dataset.csv')
df_orders_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_orders_dataset.csv')
df_products_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_products_dataset.csv')
df_sellers_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_sellers_dataset.csv')


In [ ]:
# Cria uma cópia das reviews para preservar o DataFrame original.
df_reviews_por_pedido = (
    df_order_reviews_raw.copy()
)

# Converte as datas; valores inválidos são registrados como NaT.
for coluna in ['review_creation_date', 'review_answer_timestamp']:
    df_reviews_por_pedido[coluna] = pd.to_datetime(
        df_reviews_por_pedido[coluna],
        errors='coerce'
    ).astype('datetime64[ns]')


df_reviews_por_pedido['review_score'] = (
    pd.to_numeric(
    df_reviews_por_pedido['review_score'], errors='coerce')
)

# Usa a data da resposta e, quando ausente, a data de criação da review.
df_reviews_por_pedido['review_event_timestamp'] = (
    df_reviews_por_pedido['review_answer_timestamp'].combine_first(
        df_reviews_por_pedido['review_creation_date'])
)

# Mantém notas válidas e uma única ocorrência por pedido e por review.
df_reviews_por_pedido = (
    df_reviews_por_pedido
    .dropna(subset=['order_id', 'review_score'])
    .loc[lambda df: df['review_score'].between(1, 5)]
    .sort_values(['order_id', 'review_event_timestamp'])
    .drop_duplicates(subset=['order_id'], keep='first')
    .drop_duplicates(subset=['review_id'], keep='first')
    .reset_index(drop=True)
    .copy()
)

# Indicadores booleanos permitem contar cada classe com sum().
df_reviews_por_pedido['review_ruim'] = df_reviews_por_pedido['review_score'].le(2)
df_reviews_por_pedido['review_neutra'] = df_reviews_por_pedido['review_score'].eq(3)
df_reviews_por_pedido['review_boa'] = df_reviews_por_pedido['review_score'].ge(4)

# Confirma a granularidade e a classificação das avaliações.
assert df_reviews_por_pedido['order_id'].is_unique
assert df_reviews_por_pedido['order_id'].notna().all()
assert df_reviews_por_pedido['review_score'].notna().all()
assert df_reviews_por_pedido['review_score'].between(1, 5).all()
assert (
    df_reviews_por_pedido[['review_ruim', 'review_neutra', 'review_boa']]
    .sum(axis=1)
    .eq(1)
    .all()
)


In [ ]:
# Valida as chaves antes do merge para não ocultar duplicidades.
assert df_orders_raw['order_id'].is_unique
assert df_orders_raw['order_id'].notna().all()
assert df_orders_raw['customer_id'].notna().all()
assert df_customers_raw['customer_id'].is_unique
assert df_customers_raw['customer_id'].notna().all()

# Enriquece cada pedido com a identificação e a localização do cliente.
df_pedidos_clientes = (
    df_orders_raw
    .merge(
        df_customers_raw,
        on='customer_id',
        how='left',
        validate='1:1'
    )
    .copy()
)

# Padroniza como datetime todas as datas do ciclo do pedido.
for coluna in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 
               'order_delivered_customer_date', 'order_estimated_delivery_date']:
    df_pedidos_clientes[coluna] = pd.to_datetime(
        df_pedidos_clientes[coluna],
        errors='coerce'
    ).astype('datetime64[ns]')

# Verifica que o merge preservou uma linha para cada pedido original.
assert len(df_pedidos_clientes) == len(df_orders_raw)
assert df_pedidos_clientes['order_id'].is_unique
assert df_pedidos_clientes['order_id'].notna().all()
assert df_pedidos_clientes['customer_id'].is_unique


In [ ]:
# Verifica as chaves das tabelas usadas para enriquecer os itens.
assert df_sellers_raw['seller_id'].is_unique
assert df_products_raw['product_id'].is_unique
assert df_sellers_raw['seller_id'].notna().all()
assert df_products_raw['product_id'].notna().all()
assert df_order_items_raw['order_id'].notna().all()
assert df_order_items_raw['order_item_id'].notna().all()
assert df_order_items_raw['product_id'].notna().all()
assert df_order_items_raw['seller_id'].notna().all()


# Adiciona atributos de produto e vendedor sem alterar o nível do item.
df_itens_pedidos = (
    df_order_items_raw
    .merge(
        df_products_raw,
        on='product_id',
        how='left',
        validate='m:1'
    )
    .merge(
        df_sellers_raw,
        on='seller_id',
        how='left',
        validate='m:1'
    )
    .copy()
)


df_itens_pedidos['shipping_limit_date'] = pd.to_datetime(
    df_itens_pedidos['shipping_limit_date'],
    errors='coerce'
).astype('datetime64[ns]')


# Confirma a preservação das linhas e da chave order_id + order_item_id.
assert len(df_itens_pedidos) == len(df_order_items_raw)
assert df_itens_pedidos['order_id'].notna().all()
assert df_itens_pedidos['price'].notna().all()
assert df_itens_pedidos['freight_value'].notna().all()
assert df_itens_pedidos['price'].ge(0).all()
assert df_itens_pedidos['freight_value'].ge(0).all()
assert (
    df_itens_pedidos[['order_id', 'order_item_id']]
    .duplicated()
    .sum() == 0
)


In [ ]:
# Calcula o volume cadastrado do produto em cada linha de item.
df_itens_pedidos['volume_item_cm3'] = (
    df_itens_pedidos['product_length_cm']
    * df_itens_pedidos['product_height_cm']
    * df_itens_pedidos['product_width_cm']
)

# Os indicadores preservam a informação de que existem atributos ausentes.
df_itens_pedidos['dimensoes_ausentes'] = (
    df_itens_pedidos[
        [
            'product_length_cm',
            'product_height_cm',
            'product_width_cm'
        ]
    ]
    .isna()
    .any(axis=1)
)

df_itens_pedidos['peso_ausente'] = (
    df_itens_pedidos['product_weight_g'].isna()
)

df_itens_pedidos['categoria_ausente'] = (
    df_itens_pedidos['product_category_name'].isna()
)


In [ ]:
# Consolida os diversos itens para obter exatamente uma linha por pedido.
df_resumo_itens_por_pedido = (
    df_itens_pedidos
    .groupby('order_id')
    .agg(
        quantidade_itens=(
            'order_item_id',
            'nunique'
        ),
        quantidade_produtos=(
            'product_id',
            'nunique'
        ),
        quantidade_vendedores=(
            'seller_id',
            'nunique'
        ),
        quantidade_categorias=(
            'product_category_name',
            'nunique'
        ),
        valor_produtos=(
            'price',
            'sum'
        ),
        valor_frete=(
            'freight_value',
            'sum'
        ),
        # min_count=1 evita transformar grupos totalmente ausentes em zero.
        peso_total_g=(
            'product_weight_g',
            lambda valores: valores.sum(min_count=1)
        ),
        volume_total_cm3=(
            'volume_item_cm3',
            lambda valores: valores.sum(min_count=1)
        ),
        possui_dimensoes_ausentes=(
            'dimensoes_ausentes',
            'any'
        ),
        possui_peso_ausente=(
            'peso_ausente',
            'any'
        ),
        possui_categoria_ausente=(
            'categoria_ausente',
            'any'
        )
    )
    .reset_index()
    .copy()
)

df_resumo_itens_por_pedido.head()


In [ ]:
# Cria métricas financeiras e indicadores de composição do pedido.
df_resumo_itens_por_pedido['valor_total'] = (
    df_resumo_itens_por_pedido['valor_produtos']
    + df_resumo_itens_por_pedido['valor_frete']
)

df_resumo_itens_por_pedido['pct_frete'] = (
    df_resumo_itens_por_pedido['valor_frete']
    / df_resumo_itens_por_pedido[
        'valor_produtos'
    ].replace(0, pd.NA)
)

df_resumo_itens_por_pedido['pedido_um_produto'] = (
    df_resumo_itens_por_pedido[
        'quantidade_produtos'
    ].eq(1)
)

df_resumo_itens_por_pedido['pedido_um_vendedor'] = (
    df_resumo_itens_por_pedido[
        'quantidade_vendedores'
    ].eq(1)
)

# Valida a granularidade e as quantidades geradas pela agregação.
assert df_resumo_itens_por_pedido['order_id'].is_unique
assert df_resumo_itens_por_pedido['order_id'].notna().all()

assert (
    len(df_resumo_itens_por_pedido)
    == df_itens_pedidos['order_id'].nunique()
)

assert df_resumo_itens_por_pedido[
    'quantidade_itens'
].ge(1).all()

assert df_resumo_itens_por_pedido[
    'quantidade_produtos'
].ge(1).all()

assert df_resumo_itens_por_pedido[
    'quantidade_vendedores'
].ge(1).all()


In [ ]:
# Adiciona o resumo dos itens aos pedidos, preservando inclusive pedidos sem itens.
df_pedidos = (
    df_pedidos_clientes
    .merge(
        df_resumo_itens_por_pedido,
        on='order_id',
        how='left',
        validate='one_to_one'
    )
    .copy()
)

assert df_resumo_itens_por_pedido['order_id'].is_unique
assert df_resumo_itens_por_pedido['order_id'].notna().all()

assert (
    len(df_resumo_itens_por_pedido)
    == df_itens_pedidos['order_id'].nunique()
)

assert df_resumo_itens_por_pedido[
    'quantidade_itens'
].ge(1).all()

assert df_resumo_itens_por_pedido[
    'quantidade_produtos'
].ge(1).all()

assert df_resumo_itens_por_pedido[
    'quantidade_vendedores'
].ge(1).all()

df_pedidos.head()


In [ ]:
# Acrescenta as avaliações tratadas e forma a base analítica no nível do pedido.
df_base_analitica = (
    df_pedidos
    .merge(
        df_reviews_por_pedido,
        on='order_id',
        how='left',
        validate='one_to_one'
    )
    .copy()
)

df_base_analitica['possui_itens'] = (
    df_base_analitica['quantidade_itens'].notna()
)

df_base_analitica['possui_avaliacao'] = (
    df_base_analitica['review_score'].notna()
)

df_base_analitica['pedido_entregue'] = (
    df_base_analitica['order_status'].eq('delivered')
)

df_base_analitica['delivery_delay_days'] = (
    df_base_analitica['order_delivered_customer_date']
    - df_base_analitica['order_estimated_delivery_date']
).dt.days

df_base_analitica['is_late'] = pd.Series(
    pd.NA,
    index=df_base_analitica.index,
    dtype='boolean'
)

mascara_prazo_disponivel = (
    df_base_analitica['delivery_delay_days'].notna()
)

df_base_analitica.loc[
    mascara_prazo_disponivel,
    'is_late'
] = (
    df_base_analitica.loc[
        mascara_prazo_disponivel,
        'delivery_delay_days'
    ].gt(0)
)

print('Pedidos sem itens:',df_base_analitica['possui_itens'].eq(False).sum())

print('Pedidos sem avaliação:',df_base_analitica['possui_avaliacao'].eq(False).sum())

print('Pedidos sem cálculo de atraso:',df_base_analitica['is_late'].isna().sum())

df_base_analitica.head()


In [ ]:
display(
    pd.DataFrame({
        'coluna': df_base_analitica.columns,
        'tipo': df_base_analitica.dtypes.astype(str).values,
        'quantidade_nao_nulos': df_base_analitica.notna().sum().values,
        'quantidade_nulos': df_base_analitica.isna().sum().values
    })
)


In [ ]:
display(
    pd.DataFrame({
        'coluna': df_itens_pedidos.columns,
        'tipo': df_itens_pedidos.dtypes.astype(str).values,
        'quantidade_nao_nulos': df_itens_pedidos.notna().sum().values,
        'quantidade_nulos': df_itens_pedidos.isna().sum().values
    })
)


## Exportações

Salva e confere as bases detalhadas em data/processed.

In [ ]:
CAMINHO_PROCESSADOS = RAIZ_PROJETO / 'data' / 'processed'
CAMINHO_PROCESSADOS.mkdir(parents=True, exist_ok=True)

df_base_analitica.to_parquet(
    f'{CAMINHO_PROCESSADOS}/base_analitica_pedidos.parquet',
    index=False
)

df_itens_pedidos.to_parquet(
    f'{CAMINHO_PROCESSADOS}/itens_pedidos.parquet',
    index=False
)
pd.testing.assert_frame_equal(
    df_base_analitica, pd.read_parquet(CAMINHO_PROCESSADOS / 'base_analitica_pedidos.parquet'),
    check_exact=True
)
pd.testing.assert_frame_equal(
    df_itens_pedidos, pd.read_parquet(CAMINHO_PROCESSADOS / 'itens_pedidos.parquet'),
    check_exact=True
)


In [ ]:
# Exporta as duas bases também em CSV, sem salvar o índice.
# UTF-8 com BOM facilita a leitura dos acentos no Excel e no Power BI.
PASTA_CSV = RAIZ_PROJETO / 'outputs' / 'tabelas' / 'csv'
PASTA_CSV.mkdir(parents=True, exist_ok=True)

for nome, tabela in {
    'base_analitica_pedidos': df_base_analitica,
    'itens_pedidos': df_itens_pedidos
}.items():
    caminho = PASTA_CSV / f'{nome}.csv'
    tabela.to_csv(caminho, index=False, encoding='utf-8-sig')
    print(f'CSV salvo: {caminho} ({len(tabela):,} linhas)')


CSV salvo: C:\Dataset\ecommerce\outputs\tabelas\csv\base_analitica_pedidos.csv (99,441 linhas)
CSV salvo: C:\Dataset\ecommerce\outputs\tabelas\csv\itens_pedidos.csv (112,650 linhas)
